# 01 - Solver validation

The five sanity checks of §3.7, which are §11.2 steps 1-5, plus the two things that have
to hold before any of them mean anything: the configuration is internally consistent, and
the velocity-to-displacement deconvolution is conditioned across the whole operating band.

§11.3 lists the solver sanity checks under *never cut, regardless of time*. The reason is
not diligence for its own sake. A surrogate trained against an unvalidated solver is a
surrogate for the wrong operator, and every metric downstream -- relative L2, phase error,
inversion success rate -- will look fine while being an accurate report on the wrong
physics. Nothing later in the pipeline can detect that. This notebook is the only place
that can.

**Runtime.** Checks 1-3 are seconds. Checks 4 and 5 refine the grid and sweep a radius, so
they want a GPU: budget 20-40 minutes on an A100. `run_all` skips them on CPU, and this
notebook says so loudly rather than quietly reporting three passes out of five.

**Platform.** No Modal API calls appear in this notebook or any of the other five;
`bootstrap.setup()` works out where it is running. The headless equivalent of this
notebook is

```
modal run modal_app.py::solver_checks
```

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

## The configuration is the single source of truth

`config.self_check()` prints the grid, CFL, band, material, dispersion, parameter, ring
and inversion tables and then *asserts* every claim in them. It is not a report; it is a
refusal to continue if the numbers in §2-§8 and the numbers in the code have drifted apart.

Two values in the output are deliberate deviations from the document, and both are worth
recognising when they scroll past:

- The CFL number is **derived** from the 4th-order stencil's stability limit
  (`CFL_LIMIT_4TH = 0.6061`) times the stated 0.9 safety factor, not the document's quoted
  `0.6 dx/c`. That moves `NT` from 1280 to 1408 for the same 24 T_p window. Honouring the
  safety factor and honouring the quoted number are not the same thing, and the safety
  factor is the one that keeps the run stable.
- Everything is sized on the **shear** wavelength at the worst Poisson ratio (nu = 0.37),
  never on lambda_p. lambda_s is the shorter wave, so it sets points-per-wavelength,
  dispersion and the absorber thickness.

In [ ]:
cfg.self_check()

In [ ]:
from src.models.fno2d import band_in_modes

k_needed = band_in_modes()
print(f"parameter budget (primary variant)")
print(f"  config.total_params(d_v={cfg.D_V}, kmax={cfg.KMAX}, "
      f"n_blocks={cfg.N_BLOCKS}) = {cfg.total_params():,}")
print(f"\nmode truncation")
print(f"  band top f = {max(cfg.FREQS)/cfg.FC:.3f} f_c at nu = {min(cfg.NU_LIST)} needs "
      f"mode index {k_needed:.1f}")
print(f"  KMAX = {cfg.KMAX}  ->  {cfg.KMAX/k_needed:.2f}x headroom for near-field content")
print(f"  Nyquist on the {cfg.N_NET}^2 grid is {cfg.K_NYQUIST}, so the retained band is "
      f"strictly inside the resolved band")

## The deconvolution has to be conditioned, or steps 1-5 are measuring noise

The solver is a velocity-stress FDTD and runs a DFT inside the time loop, so what comes
out is a *velocity* phasor. Everything downstream -- the network's targets, the incident
cache, the inversion's data -- is a *displacement* phasor. The bridge is

    u_hat(omega) = v_hat(omega) / (i omega s_hat(omega))

and dividing by `s_hat` is a deconvolution. The 5-cycle Hann-windowed tone burst has
spectral nulls at 0.6 f_c and 1.4 f_c; the band runs 0.66 to 1.34 f_c, which sits *inside*
those nulls but not far inside. `conditioning_report` measures how much each line is
amplified relative to the strongest one, and `assert_conditioned` refuses anything worse
than `MAX_DECONV_AMPLIFICATION = 25`.

This is one of the three named failure modes of the time-harmonic reduction (the others
being quadrature mismatch, handled by the midpoint rule at t = (n + 1/2) dt, and
wrap-around, which check 3 and the tail-energy diagnostic in notebook 02 cover). It is
checked first because a band-edge line amplified 200x would turn every subsequent number
in this notebook into a report on round-off.

In [ ]:
from src.solver import harmonic as H

rep = H.conditioning_report()
f = _np(rep["freqs"])
s = _np(rep["s_hat_abs"])
a = _np(rep["amplification"])
worst_i = int(rep["worst_index"])

print(f"{'m':>3} {'f/f_c':>8} {'|s_hat|':>12} {'amplification':>14}")
for m in range(len(f)):
    flag = "   <-- worst" if m == worst_i else ""
    print(f"{m:>3} {f[m]:8.4f} {s[m]:12.4e} {a[m]:14.3f}{flag}")

print(f"\nworst amplification {float(rep['worst']):.2f} at m = {worst_i} "
      f"(f = {f[worst_i]:.3f} f_c), limit {H.MAX_DECONV_AMPLIFICATION}")
H.assert_conditioned()
print("PASS  the deconvolution is conditioned across the whole band")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.1))

ff = np.linspace(0.3, 1.7, 601)
om = torch.tensor(2.0 * np.pi * ff, dtype=torch.float64)
sh = H.source_hat(om).abs().numpy()
ax[0].semilogy(ff, sh / sh.max(), lw=1.0, color="0.35",
               label="|s_hat| (continuous)")
ax[0].semilogy(f, s / sh.max(), "o", ms=4, color="C0", label="the 20 band lines")
for x in (0.6, 1.4):
    ax[0].axvline(x, ls=":", c="C3", lw=1.0)
ax[0].axvspan(min(f), max(f), color="C0", alpha=0.08)
ax[0].set(xlabel="f / f_c", ylabel="|s_hat| (normalised)", ylim=(1e-4, 2.0),
          title="tone-burst spectrum\n(Hann nulls at 0.6 and 1.4 f_c, dotted)")
ax[0].legend(fontsize=7.5, loc="lower center")

ax[1].plot(f, a, "o-", ms=4)
ax[1].axhline(H.MAX_DECONV_AMPLIFICATION, ls="--", c="C3", lw=1.0,
              label=f"limit = {H.MAX_DECONV_AMPLIFICATION:g}")
ax[1].set(xlabel="f / f_c", ylabel="1 / |s_hat| relative to the strongest line",
          title="deconvolution amplification per line")
ax[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "01_deconvolution_conditioning.png")
plt.show()

## §11.2 steps 1-5, and the five the review added

| # | check | gate | what a failure would mean |
|---|-------|------|---------------------------|
| 1 | energy drift with the absorber removed | `< 0.5%` over the run | the update is not conservative: wrong Lame coefficients, a stencil bug, or dt above the stability limit |
| 2 | P and S arrival times against analytic | within one time step | the wave speeds are wrong, or the source is not where the acquisition table says it is |
| 3 | residual energy after the wave has left | `< 1e-4` of peak | energy is still bouncing around inside at `t_end`; note this is *not* a reflection measurement, which is check 9 |
| 4 | Rayleigh scaling of scattered energy with radius | slope in `(3.3, 4.7)` **and** visible mode conversion | small voids are not being resolved -- the label floor is above the smallest defect in the dataset |
| 5 | grid convergence, `256^2` vs `512^2` | `< 2%` rel-L2 | the training labels are discretisation error, not physics |
| 6 | incident ring field vs the analytic Green's tensor, no fitted scale or phase | `< 5%` rel-L2 | the deconvolved A-scan is not the elastodynamic Green's function: wrong source amplitude, wrong time convention, or a Fourier sign error |
| 7 | scattered ring field vs the Pao-Mow traction-free cavity, absolutely | `< 12%` rel-L2 and `< 0.20` rad | the labels are not cavity scattering, and everything stated in those terms downstream is about a different problem |
| 8 | interface width and grid spacing, *separately* | production width in an interior minimum, and `< 2%` shift under a 3x grid refinement at fixed physical `eps` | either the interface is riding a branch of its own U, or the two refinements have been conflated |
| 9 | the same acquisition in a padded open domain | `< 2%` rel-L2 on the incident **and** the scattered field | the absorber is corrupting the labels, and the network will learn the boundary as part of the operator |
| 10 | `losses.physics_loss` read on the solver's own labels | `< 0.15` relative | the physics regulariser is charging a correct field more than the data term can outvote |

Checks 1-5 are internal consistency: the solver against itself on a finer grid, or against
a scaling law fitted to its own output. Every one of them passes for a solver that solves
the wrong problem consistently, which is what the architecture review objected to, and the
absorber is the worked example -- check 3 said `1e-4` of the energy was left in the domain
at `t_end` while check 9, once it existed, said the layer was corrupting the ring field by
22%. Energy that leaks *out* and energy that reflects *back* are different failures.

Checks 6-9 therefore compare against things outside the solver: closed-form elastodynamics
for the incident (6) and the scattered (7) field, a separated width-and-grid study (8), and
the same acquisition in a domain large enough that no reflection can return inside the
recording window (9). Check 10 points the other way -- it takes the solver as truth and
asks what the *training loss* reads on it, which is the only way to learn the floor of a
regulariser whose discretisation is not the solver's.

Check 4's gate is a conjunction, and deliberately so. The theoretical slope is 4 (energy
goes as R^4 in the 2D Rayleigh limit, amplitude as R^2), the window brackets it, and a run
that lands the slope but produces no mode conversion has almost certainly done so by
scattering off a numerical artefact rather than off the void.

In [ ]:
from src.solver import validate as V

include_slow = DEV.startswith("cuda")
if not include_slow:
    print("NO GPU DETECTED.\n"
          "Checks 4, 5, 8 and 9 need grids the production configuration does not\n"
          "use, so they will be skipped and this run reports 6 of 10.  All four are\n"
          "on the never-cut list of §11.3 -- do not quote any result from a CPU-only\n"
          "run of this notebook.\n")

t0 = time.perf_counter()
results = V.run_all(device=DEV, include_slow=include_slow)
wall = time.perf_counter() - t0

print(f"\n{len(results)} checks in {wall/60:.1f} min on {DEV}\n")
for r in results:
    print(r)

by = {r.name.split(".")[0]: r for r in results}
n_pass = sum(1 for r in results if r.passed)
print(f"\n{n_pass} / {len(results)} passed")

### Check 1 -- energy conservation

Run on a domain of doubled side (L = 16) with the absorber removed, so nothing is
deliberately absorbing and any drift is the update's own. The gate is on the *drift*
between the start and the end of the window, not on the oscillation: a staggered
velocity-stress scheme stores kinetic and strain energy half a step apart, so the discrete
total oscillates at the sampling frequency by a bounded amount that does not accumulate.
Gating the oscillation would fail a correct solver.

In [ ]:
r = by.get("1")
if r is None:
    print("check 1 not in this run")
else:
    t = _np(r.extras["t"])
    en = _np(r.extras["energy"])
    en_n = en / max(en.max(), 1e-300)
    k0 = int(en.argmax())

    fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.0))
    ax[0].plot(t, en_n, lw=0.9)
    ax[0].axvline(t[k0], ls=":", c="0.5", lw=1.0)
    ax[0].set(xlabel="t / T_p", ylabel="E / E_max",
              title=f"total energy, L = {r.extras['l_domain']:g}, no absorber")

    tail = slice(k0, None)
    ax[1].plot(t[tail], en_n[tail] / en_n[tail][0] - 1.0, lw=0.9)
    ax[1].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.1e}")
    ax[1].axhline(-r.gate, ls="--", c="C3", lw=1.0)
    ax[1].set(xlabel="t / T_p", ylabel="E(t)/E(peak) - 1",
              title=f"drift {r.value:.2e}   oscillation "
                    f"{float(r.extras['oscillation']):.2e}")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check1_energy.png")
    plt.show()

### Check 2 -- P and S arrival times

A 2-cycle burst (short enough that the P and S packets separate before either reaches the
far receivers) with arrivals picked from the analytic envelope peak, corrected for the
group delay of the burst itself, `N_c / (2 f_c)`. Picking the first threshold crossing
instead would measure the burst's rise time rather than the wave speed, and would drift
with amplitude. The peak is refined to sub-sample precision by a parabola through the
three samples around the argmax -- an integer argmax carries a +-0.5-step quantisation
error, which is half the gate.

Not every receiver can time every phase, and the three conditions that decide it are
stated from the geometry rather than from the traces:

- **the phase has to be radiated that way.** The source is a *vertical* point force, so
  its far-field P amplitude goes as `|cos t|` and its S amplitude as `|sin t|`, with `t`
  the angle between the force axis and the source-receiver line. The receivers sharing
  the source's grid row are at `t = 90` degrees exactly, so no P wave reaches them at
  all, and the argmax in their P window is picking up the S skirt. That single effect was
  the whole of the 57.5-step "P arrival error" this check used to report;
- **the two packets have to separate** -- at `nu = 1/3`, `c_s/c_p` is exactly `1/2`, so
  the peaks are `d` apart and the burst is `N_c lambda_p` long;
- **the prediction has to be in its far field.** `d/c + group delay` is a ray prediction,
  and a 2-D point source has a trailing coda as long as `d/c`, which drags the envelope
  peak late when `d/c` is not long compared with the burst. Receivers inside 2.5 burst
  lengths are reported separately instead of being scored; beyond that radius the coda is
  worth 0.05 steps, measured by propagating this burst with the exact `H_0^(2)`.

Excluded receivers are excluded, not fudged: nothing is plotted for them.

Two corrections to the *prediction* then matter more than the gate does. The offsets are
physical positions, not index differences: the force is added to `vy`, which lives on a
y-face, and `net_to_fine` lands on the lower-left fine cell of the block, so the force
sits `dx_fine/2` to the `-x` side of the nominal network cell centre. That is 0.917 steps
of P travel and 1.833 of S, on a 1.0-step gate, and it shows up as a P error swinging from
0.00 to -1.21 steps with `|cos t|` at *fixed distance* -- a direction-dependent bias, not
a speed error. `_force_xy` and `_ring_xy` are the helpers that get this right.

The gate is then read against the arrival *this discretisation* produces, not against the
continuum `d/c`. Propagating the burst through the staggered-leapfrog dispersion relation
-- fixed by the stencil coefficients `(9/8, -1/24)`, `dt`, `dx` and `N_c` before any solve
happens, with nothing fitted and no recorded trace read -- moves the predicted peak by up
to 0.83 steps for P and 1.36 for S, *early*, because at this grid's points-per-wavelength
the second-order time stepping's `+nu^2 theta^2/8` beats the fourth-order stencil's
`-0.047 theta^4` below about `1.7 f_c`. The continuum comparison is plotted alongside and
reported in `detail`, ungated: it reads 1.57 steps, so a check gated on it could not pass
on this grid, and a check that cannot pass measures nothing. The evidence that the model
is right is the residual's *flatness* -- filled markers sit at a fixed offset across every
distance and angle, where before the position fix they tracked both.

In [ ]:
r = by.get("2")
if r is None:
    print("check 2 not in this run")
else:
    d = _np(r.extras["distances"])             # all 32 receivers
    ep = _np(r.extras["errs_p"])               # length 32, NaN where not timed
    es = _np(r.extras["errs_s"])
    ep_ray = _np(r.extras["errs_p_ray"])       # same receivers, continuum d/c
    es_ray = _np(r.extras["errs_s_ray"])
    dsp_p = _np(r.extras["dispersion_p"])      # the correction itself, in steps
    dsp_s = _np(r.extras["dispersion_s"])
    up = _np(r.extras["usable_p"]).astype(bool)
    us = _np(r.extras["usable_s"]).astype(bool)
    pat_p = _np(r.extras["pattern_p"])
    pat_s = _np(r.extras["pattern_s"])
    res_ok = _np(r.extras["usable"]).astype(bool)
    asc = _np(r.extras["ascans"])

    print(f"P timed on {int(up.sum())}/{len(d)} receivers, "
          f"S on {int(us.sum())}/{len(d)}")
    print(f"  P/S packets overlap:          {int((~res_ok).sum())} receivers")
    print(f"  in the P radiation node:      {int((pat_p <= 0.3).sum())} "
          f"(|cos t| <= 0.3; {int((pat_p == 0.0).sum())} of them exactly 0)")
    print(f"  in the S radiation node:      {int((pat_s <= 0.3).sum())} (|sin t| <= 0.3)")
    print(f"  near field, P {len(r.extras['near_field_p'])} / S "
          f"{len(r.extras['near_field_s'])} receivers, worst "
          f"{max(r.extras['near_field_p'] + r.extras['near_field_s'], default=float('nan')):.2f} steps")
    print(f"worst |error| vs this scheme {r.value:.2f}, vs continuum d/c "
          f"{r.extras['worst_ray']:.2f} steps; the dispersive correction is worth up to "
          f"{np.nanmax(np.abs(np.concatenate([dsp_p, dsp_s]))):.2f}")

    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.2))
    ax[0].plot(d[up], ep_ray[up], "o", ms=5, mfc="none", c="C0", alpha=0.6,
               label="P vs continuum d/c")
    ax[0].plot(d[us], es_ray[us], "s", ms=5, mfc="none", c="C1", alpha=0.6,
               label="S vs continuum d/c")
    ax[0].plot(d[up], ep[up], "o", ms=4, c="C0", label=f"P, {int(up.sum())} recv")
    ax[0].plot(d[us], es[us], "s", ms=4, c="C1", label=f"S, {int(us.sum())} recv")
    ax[0].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:g} step")
    ax[0].set(xlabel="source-receiver distance / lambda_p",
              ylabel="|arrival error| (time steps)",
              xlim=(0.0, 1.02 * float(d.max())), ylim=(0.0, None),
              title=f"filled: this scheme, worst {r.value:.3f} steps\n"
                    f"open: continuum d/c, worst {r.extras['worst_ray']:.2f}")
    ax[0].legend(fontsize=7.0, ncol=2)

    # a few traces with their envelopes, to show the picking is on a clean packet
    trace = asc[0] if asc.ndim == 4 else asc
    pick = np.argsort(d)[::-1][:4]
    t_ax = (np.arange(trace.shape[-1]) + 0.5) * cfg.DT
    for k, i in enumerate(pick):
        x = trace[i, 0]
        env = _np(H.envelope(torch.from_numpy(np.ascontiguousarray(x))))
        off = 1.15 * k
        norm = max(abs(x).max(), 1e-30)
        ax[1].plot(t_ax, x / norm + off, lw=0.7, c=f"C{k}")
        ax[1].plot(t_ax, env / norm + off, lw=1.0, c="0.25", alpha=0.8)
    ax[1].set(xlabel="t / T_p", ylabel="receiver (offset)", yticks=[],
              title="x-velocity and analytic envelope\n(4 most distant receivers)")
    fig.tight_layout()
    savefig(fig, "01_check2_arrivals.png")
    plt.show()

### Check 3 -- the absorber, from the inside

Residual energy in the interior long after the wave has reached the boundary, as a
fraction of the peak. The layer is a graded polynomial **sponge**, not a split-field PML:
it is only reflectionless for waves that are already propagating freely when they enter
it, which is why the dataset's rejection sampler keeps every void boundary
`BOUNDARY_KEEPOUT_LS = 1.5` shear wavelengths away from the walls -- a void inside the
absorber would scatter into a medium that is not the medium the absorber was designed for.

What this check measures is how much energy is *left inside* at `t_end`. It is not a
reflection measurement, and the two answers are capable of disagreeing by four orders of
magnitude: this gate read `1e-4` while the layer was corrupting the receiver ring by 22%.
Check 9 is the one that reads reflection.

This number is also the wrap-around margin for the running DFT: the phasors are computed
over a finite window, so anything still ringing at the end of it aliases back onto the
band. Notebook 02 measures the same quantity on the real A-scans.

In [ ]:
r = by.get("3")
if r is None:
    print("check 3 not in this run")
else:
    t = _np(r.extras["t"])
    en = _np(r.extras["energy"])
    en_n = en / max(en.max(), 1e-300)
    frac = float(r.extras["tail_fraction"])

    fig, ax = plt.subplots(figsize=(5.6, 3.1))
    ax.semilogy(t, np.maximum(en_n, 1e-16), lw=0.9)
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0e}")
    ax.axvspan(t[int(0.9 * len(t))], t[-1], color="C3", alpha=0.08,
               label="tail window")
    ax.set(xlabel="t / T_p", ylabel="E(t) / E_peak",
           title=f"absorber residual {r.value:.2e}   tail fraction {frac:.2e}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check3_absorber.png")
    plt.show()

### Check 4 -- the Rayleigh limit and mode conversion

Scattered energy against void radius on log-log axes. In the 2D Rayleigh regime
(`kR << 1`) the scattered amplitude goes as R^2 and the energy as R^4, so the slope of
`log E` against `log R` should be 4. The gate window `(3.3, 4.7)` is wide because the
largest radii in the sweep are leaving the Rayleigh regime and the smallest are approaching
the label floor from check 5. `extras["local_slope"]` is the slope of the smallest pair
alone; the middle panel below recomputes every consecutive pair from the energies, which is
what actually shows where the clean scaling lives.

The second condition is that mode conversion is visible: a P wave hitting a traction-free
curved boundary must produce an S wave. If it does not, the boundary is not being resolved
as a boundary, and the "scattered field" being measured is a staircase artefact that
happens to scale plausibly. The reported ratio is scattered energy in the S arrival window
over that in the P window, for the largest void in the sweep -- one number, not a sweep.

This check is what sets the **label floor**: the smallest radius whose scattered field is
above the discretisation error is the smallest defect the dataset can honestly contain.
`R_MIN_LS = 0.4` shear wavelengths sits above it.

In [ ]:
r = by.get("4")
if r is None:
    print("check 4 not in this run (needs a GPU)")
else:
    en = _np(r.extras["energy"])
    kR = _np(r.extras["kR"])
    slope = float(r.extras["slope"])
    # `radii` is in absolute (non-dimensional length) units, but the natural axis
    # here is radius in shear wavelengths -- and kR = 2 pi R / lambda_s gives it
    # exactly, without having to assume which nu the run used.
    rad = kR / (2.0 * np.pi)
    # These two are single numbers, not one per radius: `local_slope` is the slope
    # of the smallest pair, `mode_conversion` is one ratio over the largest void.
    loc_reported = float(r.extras["local_slope"])
    conv = float(r.extras["mode_conversion"])

    # Every consecutive pair, so the panel shows where the clean R^4 scaling lives
    # rather than only reporting the one pair that validate.py happens to return.
    lp = np.diff(np.log(en)) / np.diff(np.log(rad))
    mid = np.sqrt(rad[1:] * rad[:-1])                 # geometric midpoints

    fig, ax = plt.subplots(1, 3, figsize=(11.0, 3.1))

    ax[0].loglog(rad, en, "o-", ms=4)
    ref = en[0] * (rad / rad[0]) ** 4.0
    ax[0].loglog(rad, ref, ls="--", c="0.45", lw=1.0, label="slope 4 (Rayleigh)")
    ax[0].set(xlabel="R / lambda_s", ylabel="scattered energy",
              title=f"fitted slope {slope:.3f}\ngate window (3.3, 4.7)")
    ax[0].legend(fontsize=8)

    ax[1].semilogx(mid, lp, "o-", ms=4, label="consecutive pairs")
    ax[1].axhspan(3.3, 4.7, color="C2", alpha=0.12, label="gate window")
    ax[1].axhline(4.0, ls=":", c="0.4", lw=1.0)
    ax[1].plot([mid[0]], [loc_reported], "x", c="C3", ms=9, mew=1.6,
               label=f"reported {loc_reported:.2f}")
    ax[1].set(xlabel="R / lambda_s (pair midpoint)", ylabel="d log E / d log R",
              title="local slope, pair by pair")
    ax[1].legend(fontsize=7.5)

    # One number, so one bar.  A line plot of a scalar against the four kR values
    # would suggest a sweep that was never run.
    ax[2].bar([0], [conv], width=0.55, color="C0", alpha=0.85)
    ax[2].axhline(0.01, ls="--", c="C3", lw=1.0, label="gate: > 1%")
    ax[2].set(xticks=[0], xticklabels=[f"R = {rad[-1]:.3f} lambda_s"],
              ylabel="S-window energy / P-window energy",
              yscale="log", xlim=(-0.6, 0.6),
              title=f"mode conversion {conv:.1%}\nkR in "
                    f"[{kR.min():.2f}, {kR.max():.2f}]")
    ax[2].legend(fontsize=8)

    fig.tight_layout()
    savefig(fig, "01_check4_rayleigh.png")
    plt.show()
    print(f"grid {r.extras['grid']}  dx {float(r.extras['dx']):.5f}  "
          f"nt {int(r.extras['nt'])}")

### Check 5 -- grid convergence

The same scattering problem at `256^2` and at `512^2`, compared per frequency on the
network grid. This is the number that says the training labels are physics rather than
discretisation error, and it is reported per frequency because the error is not uniform
across the band: the top of the band has the fewest points per wavelength and converges
last. If any single line is above the gate, that line's labels are not trustworthy even if
the band average passes.

In [ ]:
r = by.get("5")
if r is None:
    print("check 5 not in this run (needs a GPU)")
else:
    pf = _np(r.extras["per_frequency"]).reshape(-1)
    fr = np.asarray(cfg.FREQS[:len(pf)])

    fig, ax = plt.subplots(figsize=(6.4, 3.1))
    ax.bar(fr, pf, width=0.9 * cfg.DF, color="C0", alpha=0.85)
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax.set(xlabel="f / f_c", ylabel="rel-L2, 256^2 vs 512^2",
           title=f"grid convergence at R = {float(r.extras['radius']):.3f}, "
                 f"refine {int(r.extras['refine'])}x\nworst line {pf.max():.4f}, "
                 f"reported {r.value:.4f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check5_convergence.png")
    plt.show()

### Check 6 -- the incident field against the analytic Green's tensor

The first check with an external answer. Checks 1-5 compare the solver against itself; this
one compares the deconvolved A-scan against `cavity.green_displacement` for a unit `+y`
point force, with **no fitted amplitude and no fitted phase**, because a fitted scale would
forgive exactly the errors the check exists to find.

`rel_l2_conj` is reported beside it and is the convention test: the code takes
`exp(-i omega t)`, so `d/dt` is `+i omega` and outgoing waves are Hankel functions of the
second kind. If that were backwards, the conjugated comparison would be the small number.

Two solves, because the production geometry measures two things at once. Sources sit
`RING_INSET_NET = 3` network cells from the absorber, so a production source radiates its
near field straight into the layer at every angle including grazing; the gate is applied to
a centre-source solve and the production number reported beside it. Both are now a couple
of percent, and the ordering has flipped from what it was with the thin absorber (23%
production against 3.6% centred): with a thick layer the centred comparison is the worse of
the two, because it is dominated by the longer propagation path and its numerical dispersion
rather than by the boundary.

In [ ]:
r = by.get("6")
if r is None:
    print("check 6 not in this run")
else:
    rep, prod = r.extras["report"], r.extras["report_production"]
    pf = _np(rep["rel_l2_per_freq"]).reshape(-1)
    pfp = _np(prod["rel_l2_per_freq"]).reshape(-1)
    fr = np.asarray(cfg.FREQS[:len(pf)])

    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.1))
    ax[0].plot(fr, pf, "o-", ms=3.5, lw=1.0, label="centre source (gated)")
    ax[0].plot(fr, pfp, "s-", ms=3.5, lw=1.0, label="production source")
    ax[0].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax[0].set(xlabel="f / f_c", ylabel="rel-L2 vs Green's tensor",
              title=f"absolute error, no fitted scale\ncentre {rep['rel_l2']:.2%}, "
                    f"production {prod['rel_l2']:.2%}")
    ax[0].legend(fontsize=7.5)

    keys = ["rel_l2", "rel_l2_cal", "rel_l2_conj"]
    lbl = ["absolute", "scale+phase\ncalibrated", "conjugated\nconvention"]
    ax[1].bar(range(3), [float(rep[k]) for k in keys], width=0.6,
              color=["C0", "C0", "C3"], alpha=0.85)
    ax[1].axhline(r.gate, ls="--", c="C3", lw=1.0)
    ax[1].set(xticks=range(3), xticklabels=lbl, ylabel="rel-L2", yscale="log",
              title=f"amp ratio {float(rep['amp_ratio']):.4f}, phase "
                    f"{float(rep['phase_rad']):+.3f} rad")
    ax[1].tick_params(axis="x", labelsize=7.5)
    fig.tight_layout()
    savefig(fig, "01_check6_green.png")
    plt.show()

### Check 7 -- the labels really are cavity scattering

The claim this check exists to test. Everything downstream is stated in cavity terms -- the
physics residual, the Rayleigh scaling of check 4, the whole shape-recovery premise -- and
until this was measured it was an assumption about a soft inclusion with a smoothed
boundary. The comparison is against the Pao-Mow series for a traction-free circular cavity,
absolutely, at three radii spanning the specified range.

Two things had to change before it passed, and neither works alone. At `R = 1.2 lambda_s`:

| | `eps` = 1.5 net cells | `eps` = 0.375 fine cells |
|---|---|---|
| `rho_void = RHO0` | 87.4% (amp 0.52) | 76.7% (amp 0.71) |
| `rho_void = 1e-2 RHO0` | 86.3% (amp 0.36) | **9.3%** (amp 0.99) |

The amplitude ratios are the tell: three of the four corners return between a third and
three quarters of the right scattered amplitude, so they are not slightly wrong cavities,
they are different scatterers.

What the gate is *not* is a claim that the labels are accurate to 12%. About 3.4% of the 9.3%
is accounted for by the illumination error of check 6, the absorber term of check 9, the
circle's area on a fixed grid and the A-scan sampling model; the rest is the void model
itself, and check 8 is the evidence that it is not a discretisation. Roughly 9% is what it
costs to represent a traction-free boundary as a soft light inclusion. The phase -- the
travel-time information the inversion actually uses -- comes out an order of magnitude
inside its own gate.

In [ ]:
r = by.get("7")
if r is None:
    print("check 7 not in this run")
else:
    reps = r.extras["reports"]
    rls = np.asarray(r.extras["radii_ls"], dtype=float)
    errs = np.asarray(r.extras["rel_l2"], dtype=float)

    fig, ax = plt.subplots(1, 3, figsize=(11.4, 3.1))
    ax[0].bar(range(len(errs)), errs, width=0.55, color="C0", alpha=0.85)
    ax[0].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax[0].set(xticks=range(len(errs)),
              xticklabels=[f"{v:.2f}" for v in rls],
              xlabel="R / lambda_s", ylabel="rel-L2 vs Pao-Mow",
              title=f"absolute, worst {errs.max():.2%}")
    ax[0].legend(fontsize=8)

    fr = np.asarray(cfg.FREQS)
    for i, rep in enumerate(reps):
        pf = _np(rep["rel_l2_per_freq"]).reshape(-1)
        ax[1].plot(fr[:len(pf)], pf, "o-", ms=3.0, lw=1.0,
                   label=f"R = {rls[i]:.2f} lambda_s")
    ax[1].axhline(r.gate, ls="--", c="C3", lw=1.0)
    ax[1].set(xlabel="f / f_c", ylabel="rel-L2", title="per frequency")
    ax[1].legend(fontsize=7.5)

    amp = [float(x["amp_ratio"]) for x in reps]
    ph = [float(x["phase_rad"]) for x in reps]
    ax[2].plot(rls, amp, "o-", ms=4, lw=1.0, c="C0", label="amp ratio")
    ax[2].axhline(1.0, ls=":", c="0.5", lw=1.0)
    a2 = ax[2].twinx()
    a2.plot(rls, ph, "s--", ms=4, lw=1.0, c="C1", label="phase")
    a2.axhline(cfg.GATE_CAVITY_PHASE_RAD, ls="--", c="C3", lw=1.0)
    a2.set_ylabel("phase error / rad")
    a2.set_ylim(0.0, 1.05 * cfg.GATE_CAVITY_PHASE_RAD)
    ax[2].set(xlabel="R / lambda_s", ylabel="amplitude ratio",
              title=f"calibration; phase gate {cfg.GATE_CAVITY_PHASE_RAD:.2f} rad")
    fig.tight_layout()
    savefig(fig, "01_check7_cavity.png")
    plt.show()

### Check 8 -- interface width and grid spacing, separated

Two refinements that a single "make the grid finer" study would conflate. A smoothed
interface is a *physical* deviation from a traction-free cavity and gets better as `eps`
shrinks; a sigmoid narrower than the cell that samples it is a *discretisation* error and
gets worse. Sweeping `eps` and `dx` together cannot tell them apart.

**Part A** holds `dx` fixed and varies `eps`. Measured, 10-90% width in fine cells against
rel-L2 at `R = 1.2 lambda_s`: 2.64 -> 15.7%, 1.98 -> 11.0%, 1.65 -> 9.3%, 1.32 -> 8.9%,
0.99 -> 11.4%. A U with a broad flat bottom, and the amplitude ratio crossing 1.0 between
the last two (0.989, 1.000, 1.013) identifies the branches: a sub-cell sigmoid is a
staircase, and a staircase over-scatters. The true minimum is one step narrower than
production and is deliberately not taken -- 0.4 points of a residual dominated by the void
model, bought by moving towards the staircase branch and narrowing what the network grid has
to represent.

**Part B** holds `eps` as a *length* and refines `dx`, with `n_pml`, `n_total`, `nt` and
`downsample` all scaled so `n_net` and the physical positions of the source and the ring are
unchanged -- otherwise the two runs differ in acquisition geometry as well as grid. The
error is flat to a fraction of a point across a 3x refinement, and that is the finding: at
this width the discretisation is not the limitation, so the remaining cavity error of check
7 is the void model, not the mesh.

In [ ]:
r = by.get("8")
if r is None:
    print("check 8 not in this run (needs a GPU)")
else:
    w = np.asarray(r.extras["width_fine_cells"], dtype=float)
    errs = np.asarray(r.extras["rel_l2"], dtype=float)
    amp = np.asarray([float(x["amp_ratio"]) for x in r.extras["reports"]])
    i_prod = list(r.extras["factors"]).index(1.0)
    ref = int(r.extras["refine"])

    fig, ax = plt.subplots(1, 3, figsize=(11.4, 3.1))
    o = np.argsort(w)
    ax[0].plot(w[o], errs[o], "o-", ms=4, lw=1.1)
    ax[0].plot([w[i_prod]], [errs[i_prod]], "*", ms=13, c="C3",
               label="production width")
    ax[0].set(xlabel="10-90% interface width / fine cells", ylabel="rel-L2 vs Pao-Mow",
              title="part A: eps at fixed dx")
    ax[0].legend(fontsize=8)

    ax[1].plot(w[o], amp[o], "o-", ms=4, lw=1.1, c="C1")
    ax[1].axhline(1.0, ls=":", c="0.5", lw=1.0)
    ax[1].plot([w[i_prod]], [amp[i_prod]], "*", ms=13, c="C3")
    ax[1].set(xlabel="10-90% interface width / fine cells", ylabel="amplitude ratio",
              title="crossing 1.0 marks the staircase branch")

    pair = [errs[i_prod], float(r.extras["rel_l2_refined"])]
    ax[2].bar([0, 1], pair, width=0.55, color=["C0", "C2"], alpha=0.85)
    ax[2].axhline(pair[0] + r.gate, ls="--", c="C3", lw=1.0,
                  label=f"shift gate {r.gate:.0%}")
    ax[2].axhline(max(pair[0] - r.gate, 0.0), ls="--", c="C3", lw=1.0)
    ax[2].set(xticks=[0, 1], xticklabels=["dx", f"dx / {ref}"], ylabel="rel-L2",
              title=f"part B: dx at fixed physical eps\nshift {r.value:.2%}")
    ax[2].legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check8_width.png")
    plt.show()

### Check 9 -- what the absorber actually leaves on the receiver ring

Check 3 says how much energy is still inside the domain at `t_end`. That is not the same
question as how much came *back*, and the two answers disagreed by four orders of magnitude:
`1e-4` of the energy left inside, while the layer was corrupting the ring field by 22%.

The measurement is not a normal-incidence reflection coefficient. It differences the
production acquisition against **the same acquisition in a padded open domain** -- 208 extra
fine cells on every side, so any boundary reflection has 13 extra length units to travel and
cannot return inside the recording window. Sources sit 3 network cells from the layer, so it
is struck at every angle out to grazing, which is where a graded sponge is worst, and the
comparison reads whatever that costs rather than what a plane-wave formula predicts.

Both fields are gated. With a bad layer the incident error is much the larger, because the
scattered field is built as a difference of two solves sharing the source and the domain and
the common-mode reflection cancels; once that cancellation has nothing large left to remove,
the scattered field is the smaller signal and the same absolute artefact reads as a larger
relative one. The ordering flips, and gating only the incident field would miss it.

The absorber that made this pass is 60 fine cells at `p = 4` and a design `R_target` of
`3e-2` -- `1.24 lambda_p` at the lowest frequency, and a 1.42x cost on every solve in the
project. What the sweeps found: thickness in wavelengths dominates and saturates near
`1.2 lambda_p(f_lo)`; the grading order buys about 2x and then saturates; and `R_target` has
an **interior** minimum, because `d0 ~ -ln(R_target)` steepens the very gradient an unmatched
layer reflects while the round-trip attenuation is `R_target` regardless. The old
configuration sat on the wrong side of that minimum.

In [ ]:
r = by.get("9")
if r is None:
    print("check 9 not in this run (needs a GPU)")
else:
    pf = np.asarray(r.extras["per_freq"], dtype=float)
    fr = np.asarray(cfg.FREQS[:len(pf)])
    inc, sca = float(r.extras["incident"]), float(r.extras["scattered"])

    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.1))
    ax[0].plot(fr, pf, "o-", ms=3.5, lw=1.0)
    ax[0].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax[0].set(xlabel="f / f_c", ylabel="rel-L2 vs open domain",
              title=f"incident field, per frequency\n"
                    f"{int(r.extras['n_absorber_fine'])} cells = "
                    f"{float(r.extras['thickness_lambda_p_lo']):.2f} lambda_p(f_lo), "
                    f"p = {float(r.extras['order']):g}")
    ax[0].legend(fontsize=8)

    ax[1].bar([0, 1], [inc, sca], width=0.55, color=["C0", "C1"], alpha=0.85)
    ax[1].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax[1].set(xticks=[0, 1], xticklabels=["incident", "scattered"], ylabel="rel-L2",
              title=f"both are gated\npad {int(r.extras['pad'])} cells, "
                    f"design R = {float(r.extras['r_target']):g}")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check9_absorber.png")
    plt.show()

### Check 10 -- what the training loss reads on a correct field

Every other check on this page treats the solver as the thing under test. This one treats it
as truth and tests the *loss*. The physics residual and the solver do not share a
discretisation -- the residual is a nested pair of 4th-order centred differences of the
time-harmonic Navier operator on the `128^2` network grid, the label came from a staggered
velocity-stress scheme on the `376^2` fine grid -- so nothing forces the residual to vanish
on a solution of the solver, and a regulariser with an unmeasured floor is one that can be
trading physics for numerics.

Four rows in one batch: a homogeneous medium, then three radii. The homogeneous row is the
control that separates the two contributions, because it has no interface anywhere. Measured
0.033 homogeneous and 0.044 / 0.047 / 0.060 at `R = 0.4 / 0.8 / 1.2 lambda_s`, so about a
third of what a perfect prediction is charged is the *scheme* and it is genuinely
irreducible. The interface adds `+0.027` at the largest radius and scales with perimeter,
which is why the gate carries headroom for the transfer families rather than sitting just
above this table.

The second bar in each pair restores the interface band that the default solid-fraction
weight discards -- the other half of the review's objection, since the traction-free
condition lives exactly where the default weight is zero. It costs `+0.016` more,
monotonically in radius: the band is the worst-conditioned place in the domain for a centred
stencil, which is both why the default drops it and why dropping it is a real omission.
Neither choice is free, and `losses.make_context`'s `interface_weight` exists so the trade is
a number rather than an assumption.

In [ ]:
r = by.get("10")
if r is None:
    print("check 10 not in this run")
else:
    fl = np.asarray(r.extras["floor"], dtype=float)
    per = np.asarray(r.extras["per_radius"], dtype=float)
    rls = np.asarray(r.extras["radii_ls"], dtype=float)
    lbl = ["homogeneous"] + [f"R = {v:.2f}" for v in rls]
    dflt = np.concatenate([fl[:1], per[:, 0]])
    band = np.concatenate([fl[1:2], per[:, 1]])
    x = np.arange(len(lbl))

    fig, ax = plt.subplots(figsize=(6.8, 3.2))
    ax.bar(x - 0.19, dflt, width=0.36, color="C0", alpha=0.9,
           label="default weight (trained)")
    ax.bar(x + 0.19, band, width=0.36, color="C1", alpha=0.9,
           label=f"interface band, w = {float(r.extras['interface_weight']):g}")
    ax.axhline(fl[0], ls=":", c="0.4", lw=1.0, label="discretisation floor")
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:g}")
    ax.set(xticks=x, xticklabels=lbl, ylabel="relative physics residual",
           title=f"losses.physics_loss on the solver's own labels\n"
                 f"worst gated {r.value:.3f}, floor {fl[0]:.3f}")
    ax.tick_params(axis="x", labelsize=8)
    ax.legend(fontsize=7.5)
    fig.tight_layout()
    savefig(fig, "01_check10_physics_floor.png")
    plt.show()

## Gate summary

Written to `E.results` so notebook 06's thesis table can quote it without re-running the
slow checks. A `False` here is a stop, not a note: nothing downstream is meaningful until
it is a `True`.

In [ ]:
rows = [(r.name, f"{r.value:.4e}", f"{r.gate:.2e}", r.units or "-",
         "PASS" if r.passed else "FAIL") for r in results]
table(rows, ["check", "value", "gate", "units", ""])

record = {
    "device": DEV,
    "gpu": E.gpu_name,
    "include_slow": include_slow,
    "wall_minutes": wall / 60.0,
    "deconvolution": {"worst_amplification": float(rep["worst"]),
                      "worst_index": worst_i,
                      "limit": H.MAX_DECONV_AMPLIFICATION},
    "checks": {r.name: dict(passed=bool(r.passed), value=float(r.value),
                            gate=float(r.gate), units=r.units, detail=r.detail)
               for r in results},
    "n_pass": n_pass,
    "n_total": len(results),
}
dump(record, "01_solver_validation.json")

if not include_slow:
    print("\nINCOMPLETE: checks 4 and 5 were skipped.  Re-run on a GPU.")
elif n_pass == len(results):
    print("\nAll five solver checks pass.  The dataset in notebook 02 is worth "
          "generating.")
else:
    print("\nSTOP.  Fix the solver before generating a dataset -- a surrogate "
          "trained\non these labels would be an accurate model of the wrong "
          "operator.")

## If a gate fails

- **1 (energy).** Check `dt` against `CFL_LIMIT_4TH * CFL_SAFETY * dx / c_p` first; a
  marginally unstable run drifts slowly rather than exploding. Then check the Lame
  coefficients: `lame_from_nu` assumes `rho = c_p = 1`.
- **2 (arrivals).** Almost always the source position. `source_position` documents a
  deliberate `dx_fine/2` offset between the network-grid index and the physical
  coordinate; using the wrong one shifts every arrival by half a fine cell.
- **3 (absorber energy).** This one says how much energy is *left inside* at `t_end`, which
  is not the same failure as reflection -- see check 9. If only this gate fails, `t_end` is
  short relative to the slowest round trip, or the profile is so steep that it reflects
  instead of absorbing.
- **4 (Rayleigh).** If the slope is right but conversion is absent, the interface width
  `EPS_INTERFACE_FINE_CELLS` is too large relative to the radius being tested; if both fail
  at small R only, that is the label floor and `R_MIN_LS` needs raising.
- **5 (convergence).** Failing only at the top of the band means the network grid is
  under-resolved there; that is an argument for trimming `M_FREQ`, which notebook 02's
  size projection also suggests as its first lever.
- **6 (Green's function).** If `rel_l2_conj` is the *smaller* number, the Fourier or
  outgoing-wave convention is backwards -- fix that before reading anything else, because
  check 7 rests on it. If the absolute error is large but the fitted amplitude is 1 and the
  phase is small, it is dispersion at the top of the band, not a bug.
- **7 (cavity).** Do not widen the interface to make this pass; check 8 measures that the
  width is already at the bottom of its own U. The residual that remains is the void
  *model* -- a soft light inclusion is not a cavity -- and the only real fix is a
  traction-free boundary condition on a cut cell.
- **8 (width and grid, separated).** Part A failing at an endpoint means the production
  width is riding a branch rather than sitting in the minimum. Part A and part B moving
  together means the two refinements have been conflated again: part B must hold
  `EPS_INTERFACE_PHYS` fixed *as a length* while `dx` shrinks.
- **9 (absorber vs open domain).** Measured, not guessed: thickness in wavelengths is the
  lever, and it saturates near `1.2 lambda_p(f_lo)` -- below about one wavelength no choice
  of `ABSORBER_ORDER` or `ABSORBER_R_TARGET` reaches the gate. Raise `N_ABSORBER_NET`
  first. The order buys about 2x and then saturates, and `ABSORBER_R_TARGET` has an
  *interior* minimum, so lowering it can make things worse: `d0 ~ -ln(R_target)` steepens
  the very gradient that an unmatched layer reflects, while the round-trip attenuation is
  `R_target` regardless of thickness. Both fields are gated, and once the common-mode
  reflection is small the scattered one reads worse -- that is arithmetic, not a
  regression.
- **10 (physics residual on labels).** A third of this is the network grid and cannot be
  removed. Excess above the homogeneous control is interface; excess that appears only with
  `interface_weight` set is the stencil straddling the boundary. If it fails, suspect
  `ERODE_CELLS` or the frequency conditioning before suspecting the solver -- the labels
  are the thing this check treats as true.